# 🗺️ POI RAG System - Baseline Notebook

**目的**: Raspberry Pi 4B環境で構築したRAGシステムをGoogle Colab上で再現し、GPU高速化を実現する

**概要**:
- OpenStreetMapからPOI（Point of Interest）データを取得
- LangChain + ChromaDBでRAGシステムを構築
- RAGあり/なしの性能比較を実施

**実行時間目安**: 約30-60分（初回セットアップ含む）

---
## Section 1: 環境セットアップ (Phase A)

### 1.1 パッケージインストール

In [ ]:
%%capture
# LLM・Transformers関連
!pip install -q transformers accelerate bitsandbytes

# LangChain関連
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma

# ベクトルDB・Embedding
!pip install -q chromadb sentence-transformers

# ユーティリティ
!pip install -q tqdm pandas matplotlib japanize-matplotlib requests

print("✅ パッケージインストール完了")

### 1.2 GPU確認

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name}")
    print(f"   VRAM: {gpu_memory:.1f} GB")
else:
    print("⚠️ GPUが利用できません。ランタイムを変更してください。")
    print("   メニュー: ランタイム → ランタイムのタイプを変更 → GPU")

### 1.3 Google Driveマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Driveマウント完了")

### 1.4 設定値定義

In [ ]:
import os
from datetime import datetime

# =============================================================================
# パス設定
# =============================================================================
BASE_DIR = "/content/drive/MyDrive/experiments-local-llm"
DATA_DIR = f"{BASE_DIR}/data"
RESULTS_DIR = f"{BASE_DIR}/results"
SRC_DIR = f"{BASE_DIR}/src"

# ディレクトリ作成
for dir_path in [DATA_DIR, RESULTS_DIR, SRC_DIR]:
    os.makedirs(dir_path, exist_ok=True)

# =============================================================================
# モデル設定
# =============================================================================
LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"

# =============================================================================
# テスト設定
# =============================================================================
QUICK_TEST_COUNT = 5
FULL_TEST_COUNT = 30

# =============================================================================
# POI取得設定
# =============================================================================
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
POI_AREA = {
    "name": "渋谷駅周辺",
    "bbox": "35.655,139.695,35.665,139.710"  # south,west,north,east
}

print(f"✅ 設定完了")
print(f"   BASE_DIR: {BASE_DIR}")
print(f"   LLM_MODEL: {LLM_MODEL}")
print(f"   EMBEDDING_MODEL: {EMBEDDING_MODEL}")

---
## Section 2: データ準備 (Phase A)

### 2.1 OSMからPOIデータ取得

In [ ]:
import requests
import json
from pathlib import Path

def build_overpass_query(bbox: str) -> str:
    """Overpass APIクエリを構築"""
    return f"""
[out:json][timeout:60];
(
  // 飲食店
  node["amenity"~"restaurant|cafe|fast_food|bar|pub"]({bbox});
  way["amenity"~"restaurant|cafe|fast_food|bar|pub"]({bbox});
  
  // コンビニ・商店
  node["shop"~"convenience|supermarket|bakery|books"]({bbox});
  way["shop"~"convenience|supermarket|bakery|books"]({bbox});
  
  // 観光・娯楽
  node["tourism"~"attraction|museum|hotel|information"]({bbox});
  way["tourism"~"attraction|museum|hotel|information"]({bbox});
  node["amenity"~"cinema|theatre"]({bbox});
  
  // 交通
  node["railway"="station"]({bbox});
  node["amenity"="parking"]({bbox});
  
  // 公共施設
  node["amenity"~"hospital|clinic|pharmacy|bank|post_office|police"]({bbox});
  way["amenity"~"hospital|clinic|pharmacy|bank|post_office|police"]({bbox});
);
out center tags;
"""

def get_category(tags: dict) -> str:
    """タグからカテゴリを判定"""
    if "amenity" in tags:
        amenity = tags["amenity"]
        category_map = {
            "restaurant": "飲食店/レストラン",
            "cafe": "飲食店/カフェ",
            "fast_food": "飲食店/ファストフード",
            "bar": "飲食店/バー",
            "pub": "飲食店/パブ",
            "cinema": "娯楽/映画館",
            "theatre": "娯楽/劇場",
            "hospital": "医療/病院",
            "clinic": "医療/クリニック",
            "pharmacy": "医療/薬局",
            "bank": "金融/銀行",
            "post_office": "公共/郵便局",
            "police": "公共/警察",
            "parking": "交通/駐車場",
        }
        return category_map.get(amenity, f"施設/{amenity}")
    
    if "shop" in tags:
        shop = tags["shop"]
        shop_map = {
            "convenience": "商店/コンビニ",
            "supermarket": "商店/スーパー",
            "bakery": "商店/パン屋",
            "books": "商店/書店",
        }
        return shop_map.get(shop, f"商店/{shop}")
    
    if "tourism" in tags:
        tourism = tags["tourism"]
        tourism_map = {
            "attraction": "観光/名所",
            "museum": "観光/博物館",
            "hotel": "宿泊/ホテル",
            "information": "観光/案内所",
        }
        return tourism_map.get(tourism, f"観光/{tourism}")
    
    if "railway" in tags:
        return "交通/鉄道駅"
    
    return "その他"

def fetch_and_convert_pois(area_config: dict) -> list:
    """POIを取得してRAG用ドキュメントに変換"""
    query = build_overpass_query(area_config["bbox"])
    
    print(f"📡 Overpass APIにクエリ送信中: {area_config['name']}...")
    response = requests.post(OVERPASS_URL, data={"data": query}, timeout=120)
    response.raise_for_status()
    osm_data = response.json()
    
    elements = osm_data.get("elements", [])
    print(f"   取得した要素数: {len(elements)}")
    
    documents = []
    for element in elements:
        tags = element.get("tags", {})
        if not tags:
            continue
        
        # 名前がないPOIはスキップ
        name = tags.get("name", tags.get("name:ja", ""))
        if not name:
            continue
        
        # 座標を取得（wayの場合はcenterを使用）
        if element.get("type") == "node":
            lat = element.get("lat")
            lon = element.get("lon")
        else:
            center = element.get("center", {})
            lat = center.get("lat")
            lon = center.get("lon")
        
        if not lat or not lon:
            continue
        
        name_en = tags.get("name:en", "")
        category = get_category(tags)
        
        # 住所情報を構築
        addr_parts = []
        for key in ["addr:province", "addr:city", "addr:district", "addr:street", "addr:housenumber"]:
            if key in tags:
                addr_parts.append(tags[key])
        address = "".join(addr_parts) if addr_parts else tags.get("addr:full", "住所情報なし")
        
        # ドキュメント生成
        doc_text = f"""【POI名称】{name}
【英語名】{name_en if name_en else "なし"}
【カテゴリ】{category}
【エリア】{area_config['name']}
【座標】緯度 {lat:.6f}, 経度 {lon:.6f}
【住所】{address}
【営業時間】{tags.get('opening_hours', '営業時間情報なし')}
【電話番号】{tags.get('phone', tags.get('contact:phone', '電話番号情報なし'))}
【ウェブサイト】{tags.get('website', tags.get('contact:website', 'なし'))}
【説明】{tags.get('description', tags.get('description:ja', ''))}"""
        
        documents.append({
            "id": f"osm_{element['type']}_{element['id']}",
            "content": doc_text.strip(),
            "metadata": {
                "osm_id": element["id"],
                "osm_type": element["type"],
                "name": name,
                "name_en": name_en,
                "category": category,
                "area": area_config['name'],
                "lat": lat,
                "lon": lon,
                "source": "openstreetmap"
            }
        })
    
    print(f"   有効なPOI数: {len(documents)}")
    return documents

# POIデータ取得
print("=" * 50)
print("OSM POIデータ取得ツール")
print("=" * 50)

poi_documents = fetch_and_convert_pois(POI_AREA)

print(f"\n✅ 合計ドキュメント数: {len(poi_documents)}")

### 2.2 データ保存

In [ ]:
# POIデータをGoogle Driveに保存
poi_path = f"{DATA_DIR}/poi_documents.json"
with open(poi_path, "w", encoding="utf-8") as f:
    json.dump(poi_documents, f, ensure_ascii=False, indent=2)

print(f"✅ POIデータ保存完了: {poi_path}")
print(f"   データ件数: {len(poi_documents)}件")

### 2.3 データ確認・統計表示

In [ ]:
import pandas as pd
from collections import Counter

# カテゴリ分布を確認
categories = [doc["metadata"]["category"] for doc in poi_documents]
category_counts = Counter(categories)

print("=" * 50)
print("POIデータ統計")
print("=" * 50)
print(f"\n総POI数: {len(poi_documents)}件\n")
print("【カテゴリ別分布】")
for cat, count in sorted(category_counts.items(), key=lambda x: -x[1]):
    print(f"  {cat}: {count}件")

# サンプルデータ表示
print("\n【サンプルデータ（最初の3件）】")
for i, doc in enumerate(poi_documents[:3], 1):
    print(f"\n--- {i}. {doc['metadata']['name']} ---")
    print(doc["content"][:300] + "...")

---
## Section 3: モデルロード (Phase B)

### 3.1 LLMモデルロード

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# 量子化設定（T4 GPU用）
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print(f"Loading {LLM_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)
print(f"✅ LLMモデルロード完了: {LLM_MODEL}")

### 3.2 Embeddingモデルロード

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)
print(f"✅ Embeddingモデルロード完了: {EMBEDDING_MODEL}")

### 3.3 動作確認テスト

In [ ]:
# LLM動作確認
test_prompt = "こんにちは、渋谷駅について教えてください。"
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("✅ LLM動作確認OK")
print(f"   入力: {test_prompt}")
print(f"   出力: {response[len(test_prompt):200]}...")

# Embedding動作確認
test_embedding = embeddings.embed_query("渋谷駅")
print(f"\n✅ Embedding動作確認OK")
print(f"   次元数: {len(test_embedding)}")

---
## Section 4: RAGシステム構築 (Phase B)

### 4.1 ベクトルストア構築

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

# LangChain Documentオブジェクトに変換
documents = [
    Document(
        page_content=poi["content"],
        metadata=poi["metadata"]
    )
    for poi in poi_documents
]

print(f"ベクトルストア構築中... ({len(documents)}件)")
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="poi_shibuya"
)
print(f"✅ ベクトルストア構築完了: {len(documents)}件")

### 4.2 RAGシステムクラス定義

In [ ]:
from typing import List, Dict, Optional, Tuple
import re
import time

# カテゴリキーワードマッピング
CATEGORY_KEYWORDS = {
    "レストラン": ["飲食店/レストラン"],
    "カフェ": ["飲食店/カフェ"],
    "コーヒー": ["飲食店/カフェ"],
    "バー": ["飲食店/バー"],
    "居酒屋": ["飲食店/バー", "飲食店/パブ"],
    "ファストフード": ["飲食店/ファストフード"],
    "マクドナルド": ["飲食店/ファストフード"],
    "コンビニ": ["商店/コンビニ"],
    "ローソン": ["商店/コンビニ"],
    "セブン": ["商店/コンビニ"],
    "映画館": ["娯楽/映画館"],
    "シネマ": ["娯楽/映画館"],
    "ホテル": ["宿泊/ホテル"],
    "駅": ["交通/鉄道駅"],
    "銀行": ["金融/銀行"],
    "郵便局": ["公共/郵便局"],
    "病院": ["医療/病院"],
    "薬局": ["医療/薬局"],
    "交番": ["公共/警察"],
}

def detect_category(question: str) -> Tuple[List[str], List[str]]:
    """質問文からカテゴリを検出"""
    detected = []
    matched_keywords = []
    for keyword, categories in CATEGORY_KEYWORDS.items():
        if keyword in question:
            detected.extend(categories)
            matched_keywords.append(keyword)
    return list(set(detected)), matched_keywords


class POI_RAG_System:
    """POI検索用RAGシステム（Colab版）"""
    
    def __init__(self, llm_model, tokenizer, vectorstore, debug=False):
        self.model = llm_model
        self.tokenizer = tokenizer
        self.vectorstore = vectorstore
        self.debug = debug
    
    def _generate_response(self, prompt: str, max_tokens: int = 512) -> str:
        """LLMでレスポンスを生成"""
        messages = [
            {"role": "system", "content": "あなたは渋谷エリアの地理情報に詳しいアシスタントです。提供された情報に基づいて、正確かつ簡潔に回答してください。座標情報がある場合は必ず含めてください。"},
            {"role": "user", "content": prompt}
        ]
        
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        inputs = self.tokenizer(text, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=0.1,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # アシスタントの応答部分を抽出
        if "assistant" in response.lower():
            parts = response.split("assistant")
            if len(parts) > 1:
                response = parts[-1].strip()
        
        return response
    
    def search_only(self, query: str, k: int = 5) -> List[Dict]:
        """ベクトル検索のみ実行"""
        detected_categories, _ = detect_category(query)
        
        if detected_categories:
            results = self.vectorstore.similarity_search(query, k=k * 2)
            filtered = [r for r in results if r.metadata.get("category") in detected_categories][:k]
            if filtered:
                results = filtered
            else:
                results = results[:k]
        else:
            results = self.vectorstore.similarity_search(query, k=k)
        
        return [
            {
                "name": r.metadata.get("name", "不明"),
                "category": r.metadata.get("category", "不明"),
                "lat": r.metadata.get("lat"),
                "lon": r.metadata.get("lon"),
                "content": r.page_content
            }
            for r in results
        ]
    
    def query_with_rag(self, question: str) -> Dict:
        """RAGを使用して質問に回答"""
        start_time = time.time()
        search_results = self.search_only(question, k=5)
        context = "\n\n".join([r["content"] for r in search_results])
        
        prompt = f"""以下の情報を参考にして、質問に回答してください。

【参考情報】
{context}

【質問】
{question}

【回答】
上記の情報を基に回答します。座標情報がある場合は必ず含めてください。"""
        
        answer = self._generate_response(prompt)
        elapsed_time = time.time() - start_time
        
        return {
            "answer": answer,
            "sources": search_results,
            "time_ms": int(elapsed_time * 1000)
        }
    
    def query_without_rag(self, question: str) -> Dict:
        """RAGを使用せずに質問に回答"""
        start_time = time.time()
        prompt = f"""以下の質問に回答してください。\n\n【質問】\n{question}\n\n【回答】"""
        answer = self._generate_response(prompt)
        elapsed_time = time.time() - start_time
        
        return {
            "answer": answer,
            "time_ms": int(elapsed_time * 1000)
        }

print("✅ RAGシステムクラス定義完了")

### 4.3 RAGシステム初期化

In [ ]:
rag_system = POI_RAG_System(
    llm_model=model,
    tokenizer=tokenizer,
    vectorstore=vectorstore,
    debug=True
)
print("✅ RAGシステム初期化完了")

### 4.4 単体動作確認

In [ ]:
test_question = "渋谷駅の場所を教えてください"
print("=" * 50)
print(f"テスト質問: {test_question}")
print("=" * 50)

print("\n【RAGあり】")
rag_result = rag_system.query_with_rag(test_question)
print(f"回答: {rag_result['answer'][:500]}")
print(f"処理時間: {rag_result['time_ms']}ms")

print("\n【RAGなし】")
no_rag_result = rag_system.query_without_rag(test_question)
print(f"回答: {no_rag_result['answer'][:500]}")
print(f"処理時間: {no_rag_result['time_ms']}ms")

---
## Section 5: テストケース定義 (Phase C)

In [ ]:
from dataclasses import dataclass, asdict
from typing import List, Optional

@dataclass
class TestCase:
    id: int
    category: str
    prompt: str
    expected_keywords: List[str]
    expected_data_type: str
    difficulty: str
    expected_category: Optional[str] = None
    description: str = ""

@dataclass
class TestResult:
    test_id: int
    test_category: str
    prompt: str
    rag_answer: str
    rag_time_ms: int
    rag_keyword_hits: int
    rag_keyword_total: int
    rag_has_coordinate: bool
    rag_has_poi_name: bool
    no_rag_answer: str
    no_rag_time_ms: int
    no_rag_keyword_hits: int
    no_rag_has_coordinate: bool
    no_rag_has_poi_name: bool
    rag_score: float
    no_rag_score: float
    improvement: float

# 30件のテストケース定義
TEST_CASES: List[TestCase] = [
    TestCase(1, "location", "渋谷駅の場所を教えてください", ["渋谷", "駅", "35.", "139."], "coordinate", "easy", "交通/鉄道駅"),
    TestCase(2, "location", "東宝シネマの座標は？", ["東宝", "シネマ", "座標"], "coordinate", "easy", "娯楽/映画館"),
    TestCase(3, "location", "渋谷東武ホテルはどこにありますか？", ["東武", "ホテル", "渋谷"], "coordinate", "easy", "宿泊/ホテル"),
    TestCase(4, "location", "渋谷神南郵便局の場所を教えて", ["神南", "郵便局", "渋谷"], "coordinate", "easy", "公共/郵便局"),
    TestCase(5, "location", "マクドナルドの位置情報を教えてください", ["マクドナルド", "座標"], "coordinate", "easy", "飲食店/ファストフード"),
    TestCase(6, "location", "ローソンはどこにありますか？", ["ローソン", "座標"], "coordinate", "easy", "商店/コンビニ"),
    TestCase(7, "location", "ヒューマントラストシネマ渋谷の場所", ["ヒューマントラスト", "シネマ"], "coordinate", "medium", "娯楽/映画館"),
    TestCase(8, "location", "渋谷警察署渋谷駅前交番の位置", ["警察", "交番", "渋谷"], "coordinate", "medium", "公共/警察"),
    TestCase(9, "location", "パルコ劇場はどこですか？", ["パルコ", "劇場"], "coordinate", "medium", "娯楽/劇場"),
    TestCase(10, "location", "渋谷の三菱UFJ銀行の場所", ["三菱", "UFJ", "銀行"], "coordinate", "medium", "金融/銀行"),
    TestCase(11, "nearby", "渋谷駅周辺のコンビニを教えてください", ["コンビニ", "ローソン", "ファミリーマート"], "name", "medium", "商店/コンビニ"),
    TestCase(12, "nearby", "渋谷にあるカフェを教えて", ["カフェ", "コーヒー"], "name", "medium", "飲食店/カフェ"),
    TestCase(13, "nearby", "渋谷周辺のレストランを3つ挙げてください", ["レストラン", "店"], "name", "medium", "飲食店/レストラン"),
    TestCase(14, "nearby", "渋谷の映画館を全部教えて", ["映画館", "シネマ"], "name", "medium", "娯楽/映画館"),
    TestCase(15, "nearby", "渋谷にある薬局はどこですか？", ["薬局", "ドラッグ"], "name", "medium", "医療/薬局"),
    TestCase(16, "nearby", "渋谷周辺の銀行を教えてください", ["銀行", "ATM"], "name", "medium", "金融/銀行"),
    TestCase(17, "nearby", "渋谷駅近くのホテルはありますか？", ["ホテル", "宿泊"], "name", "medium", "宿泊/ホテル"),
    TestCase(18, "nearby", "渋谷にファストフード店はありますか？", ["ファストフード", "マクドナルド"], "name", "medium", "飲食店/ファストフード"),
    TestCase(19, "nearby", "渋谷の郵便局を教えて", ["郵便局"], "name", "medium", "公共/郵便局"),
    TestCase(20, "nearby", "渋谷にバーはありますか？", ["バー", "Bar"], "name", "medium", "飲食店/バー"),
    TestCase(21, "category_search", "渋谷で食事できる場所を教えて", ["レストラン", "カフェ", "飲食"], "name", "medium"),
    TestCase(22, "category_search", "渋谷の娯楽施設を教えて", ["映画館", "劇場", "シネマ"], "name", "medium"),
    TestCase(23, "category_search", "渋谷の金融機関を教えてください", ["銀行", "信用金庫"], "name", "medium"),
    TestCase(24, "category_search", "渋谷の公共施設を教えて", ["郵便局", "警察", "交番"], "name", "medium"),
    TestCase(25, "category_search", "渋谷で買い物できる場所は？", ["コンビニ", "スーパー", "店"], "name", "medium"),
    TestCase(26, "complex", "渋谷駅から一番近いコンビニの座標を教えて", ["コンビニ", "座標", "緯度"], "coordinate", "hard"),
    TestCase(27, "complex", "渋谷で映画を見た後に食事できる場所を教えて", ["映画館", "レストラン", "カフェ"], "name", "hard"),
    TestCase(28, "complex", "渋谷で朝食をとれる場所とその座標を教えて", ["カフェ", "ファストフード", "座標"], "coordinate", "hard"),
    TestCase(29, "complex", "渋谷駅周辺でATMと郵便局の両方がある場所", ["ATM", "銀行", "郵便局"], "name", "hard"),
    TestCase(30, "complex", "渋谷のホテルとその周辺のコンビニを教えて", ["ホテル", "コンビニ", "ローソン"], "name", "hard"),
]

print(f"✅ テストケース定義完了: {len(TEST_CASES)}件")

### 5.3 評価関数定義

In [ ]:
import re

def count_keyword_hits(answer: str, keywords: List[str]) -> int:
    hits = 0
    answer_lower = answer.lower()
    for keyword in keywords:
        if keyword.lower() in answer_lower:
            hits += 1
    return hits

def has_coordinate(answer: str) -> bool:
    patterns = [r'35\.\d{3,}', r'139\.\d{3,}', r'緯度.*\d+\.\d+', r'経度.*\d+\.\d+']
    for pattern in patterns:
        if re.search(pattern, answer, re.IGNORECASE):
            return True
    return False

def has_poi_name(answer: str, poi_documents: list) -> bool:
    for poi in poi_documents:
        name = poi["metadata"].get("name", "")
        if name and len(name) > 2 and name in answer:
            return True
    return False

def calculate_score(keyword_hits, keyword_total, has_coord, has_name, expected_data_type) -> float:
    keyword_score = (keyword_hits / keyword_total * 100) if keyword_total > 0 else 0
    coord_score = 100 if has_coord else 0
    if expected_data_type != "coordinate":
        coord_score = 50
    name_score = 100 if has_name else 0
    return round(keyword_score * 0.4 + coord_score * 0.3 + name_score * 0.3, 1)

print("✅ 評価関数定義完了")

---
## Section 6: テスト実行 (Phase D)

In [ ]:
from tqdm import tqdm

def run_single_test(rag_system, test_case: TestCase, verbose: bool = True) -> TestResult:
    if verbose:
        print(f"\n[Test {test_case.id}] {test_case.prompt[:40]}...")
    
    rag_result = rag_system.query_with_rag(test_case.prompt)
    no_rag_result = rag_system.query_without_rag(test_case.prompt)
    
    rag_keyword_hits = count_keyword_hits(rag_result["answer"], test_case.expected_keywords)
    no_rag_keyword_hits = count_keyword_hits(no_rag_result["answer"], test_case.expected_keywords)
    
    rag_has_coord = has_coordinate(rag_result["answer"])
    no_rag_has_coord = has_coordinate(no_rag_result["answer"])
    
    rag_has_name = has_poi_name(rag_result["answer"], poi_documents)
    no_rag_has_name = has_poi_name(no_rag_result["answer"], poi_documents)
    
    rag_score = calculate_score(rag_keyword_hits, len(test_case.expected_keywords), rag_has_coord, rag_has_name, test_case.expected_data_type)
    no_rag_score = calculate_score(no_rag_keyword_hits, len(test_case.expected_keywords), no_rag_has_coord, no_rag_has_name, test_case.expected_data_type)
    
    if verbose:
        print(f"  RAG: {rag_score:.1f} / NoRAG: {no_rag_score:.1f} / 改善: {rag_score - no_rag_score:+.1f}")
    
    return TestResult(
        test_id=test_case.id, test_category=test_case.category, prompt=test_case.prompt,
        rag_answer=rag_result["answer"], rag_time_ms=rag_result["time_ms"],
        rag_keyword_hits=rag_keyword_hits, rag_keyword_total=len(test_case.expected_keywords),
        rag_has_coordinate=rag_has_coord, rag_has_poi_name=rag_has_name,
        no_rag_answer=no_rag_result["answer"], no_rag_time_ms=no_rag_result["time_ms"],
        no_rag_keyword_hits=no_rag_keyword_hits, no_rag_has_coordinate=no_rag_has_coord,
        no_rag_has_poi_name=no_rag_has_name, rag_score=rag_score, no_rag_score=no_rag_score,
        improvement=rag_score - no_rag_score
    )

def run_tests(rag_system, test_cases: List[TestCase], verbose: bool = True) -> List[TestResult]:
    results = []
    for tc in tqdm(test_cases, desc="テスト実行"):
        result = run_single_test(rag_system, tc, verbose=verbose)
        results.append(result)
    return results

print("✅ テストランナー定義完了")

In [ ]:
print("=" * 60)
print(f"クイックテスト（{QUICK_TEST_COUNT}件）")
print("=" * 60)
quick_results = run_tests(rag_system, TEST_CASES[:QUICK_TEST_COUNT], verbose=True)

avg_rag = sum(r.rag_score for r in quick_results) / len(quick_results)
avg_no_rag = sum(r.no_rag_score for r in quick_results) / len(quick_results)
print(f"\n平均: RAG={avg_rag:.1f}, NoRAG={avg_no_rag:.1f}, 改善={avg_rag-avg_no_rag:+.1f}")

In [ ]:
print("\n" + "=" * 60)
print(f"フルテスト（{FULL_TEST_COUNT}件）")
print("=" * 60)
full_results = run_tests(rag_system, TEST_CASES, verbose=False)
print(f"\n✅ フルテスト完了: {len(full_results)}件")

In [ ]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

results_data = {
    "timestamp": timestamp,
    "model": LLM_MODEL,
    "embedding_model": EMBEDDING_MODEL,
    "test_count": len(full_results),
    "poi_count": len(poi_documents),
    "results": [asdict(r) for r in full_results]
}

output_path = f"{RESULTS_DIR}/baseline_result_{timestamp}.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results_data, f, ensure_ascii=False, indent=2)
print(f"✅ 結果保存: {output_path}")

---
## Section 7: 結果分析・比較 (Phase E)

In [ ]:
import pandas as pd

df = pd.DataFrame([asdict(r) for r in full_results])

print("=" * 60)
print("テスト結果サマリー")
print("=" * 60)
print(f"\n【全体統計】")
print(f"  テスト件数: {len(df)}件")
print(f"  平均RAGスコア: {df['rag_score'].mean():.1f}")
print(f"  平均NoRAGスコア: {df['no_rag_score'].mean():.1f}")
print(f"  平均改善率: {df['improvement'].mean():+.1f}")
print(f"  平均RAG処理時間: {df['rag_time_ms'].mean():.0f}ms")
print(f"  平均NoRAG処理時間: {df['no_rag_time_ms'].mean():.0f}ms")

rag_keyword_rate = df['rag_keyword_hits'].sum() / df['rag_keyword_total'].sum() * 100
no_rag_keyword_rate = df['no_rag_keyword_hits'].sum() / df['rag_keyword_total'].sum() * 100
print(f"\n【キーワードヒット率】 RAG: {rag_keyword_rate:.1f}% / NoRAG: {no_rag_keyword_rate:.1f}%")
print(f"【座標含有率】 RAG: {df['rag_has_coordinate'].mean()*100:.1f}% / NoRAG: {df['no_rag_has_coordinate'].mean()*100:.1f}%")
print(f"【POI名含有率】 RAG: {df['rag_has_poi_name'].mean()*100:.1f}% / NoRAG: {df['no_rag_has_poi_name'].mean()*100:.1f}%")

In [ ]:
# カテゴリ別分析
category_stats = df.groupby('test_category').agg({
    'rag_score': 'mean', 'no_rag_score': 'mean', 'improvement': 'mean', 'rag_time_ms': 'mean', 'test_id': 'count'
}).rename(columns={'test_id': 'count'})

print("\n【カテゴリ別スコア】")
for cat in category_stats.index:
    row = category_stats.loc[cat]
    print(f"  {cat}: RAG={row['rag_score']:.1f}, NoRAG={row['no_rag_score']:.1f}, 改善={row['improvement']:+.1f}")

In [ ]:
# Raspberry Pi比較
raspi_results = {
    "総合スコア_RAGあり": 68.1, "総合スコア_RAGなし": 61.4,
    "平均応答時間_RAGあり_ms": 318000, "改善率": 10.9
}

colab_results = {
    "総合スコア_RAGあり": df['rag_score'].mean(),
    "総合スコア_RAGなし": df['no_rag_score'].mean(),
    "平均応答時間_RAGあり_ms": df['rag_time_ms'].mean(),
    "改善率": df['improvement'].mean()
}

speedup = raspi_results['平均応答時間_RAGあり_ms'] / colab_results['平均応答時間_RAGあり_ms']

print("\n" + "=" * 60)
print("Raspberry Pi 4B vs Google Colab 比較")
print("=" * 60)
print(f"\nRAGスコア: RasPi={raspi_results['総合スコア_RAGあり']:.1f} / Colab={colab_results['総合スコア_RAGあり']:.1f}")
print(f"処理時間: RasPi={raspi_results['平均応答時間_RAGあり_ms']/1000:.0f}秒 / Colab={colab_results['平均応答時間_RAGあり_ms']/1000:.1f}秒")
print(f"\n🚀 速度向上: {speedup:.1f}倍")

In [ ]:
import matplotlib.pyplot as plt
import japanize_matplotlib

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. カテゴリ別スコア比較
ax1 = axes[0, 0]
cats = category_stats.index.tolist()
x = range(len(cats))
w = 0.35
ax1.bar([i-w/2 for i in x], category_stats['rag_score'], w, label='RAGあり', color='steelblue')
ax1.bar([i+w/2 for i in x], category_stats['no_rag_score'], w, label='RAGなし', color='lightcoral')
ax1.set_xticks(x)
ax1.set_xticklabels(cats, rotation=45, ha='right')
ax1.set_title('カテゴリ別スコア比較')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# 2. 改善率
ax2 = axes[0, 1]
colors = ['green' if v > 0 else 'red' for v in category_stats['improvement']]
ax2.bar(cats, category_stats['improvement'], color=colors)
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.set_xticklabels(cats, rotation=45, ha='right')
ax2.set_title('カテゴリ別改善率')
ax2.grid(axis='y', alpha=0.3)

# 3. 処理時間比較
ax3 = axes[1, 0]
time_data = {'RasPi(RAG)': 318, 'Colab(RAG)': colab_results['平均応答時間_RAGあり_ms']/1000}
ax3.bar(time_data.keys(), time_data.values(), color=['coral', 'steelblue'])
ax3.set_ylabel('処理時間 (秒)')
ax3.set_title('処理時間比較')
ax3.grid(axis='y', alpha=0.3)

# 4. スコア散布図
ax4 = axes[1, 1]
ax4.scatter(df['no_rag_score'], df['rag_score'], alpha=0.7)
ax4.plot([0, 100], [0, 100], 'r--', label='y=x')
ax4.set_xlabel('NoRAGスコア')
ax4.set_ylabel('RAGスコア')
ax4.set_title('個別テストスコア')
ax4.legend()
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/baseline_analysis_{timestamp}.png', dpi=150)
plt.show()
print(f"\n✅ グラフ保存完了")

In [ ]:
# レポート生成
report = f"""# POI RAG System - Baseline Test Report

**実行日時**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**環境**: Google Colab (GPU: T4)

## 1. 設定
- LLM: {LLM_MODEL}
- Embedding: {EMBEDDING_MODEL}
- POI: {len(poi_documents)}件
- テスト: {len(TEST_CASES)}件

## 2. 結果
| 指標 | RAGあり | RAGなし |
|------|---------|--------|
| スコア | {colab_results['総合スコア_RAGあり']:.1f} | {colab_results['総合スコア_RAGなし']:.1f} |
| 処理時間 | {colab_results['平均応答時間_RAGあり_ms']:.0f}ms | - |

## 3. Raspberry Pi比較
- 速度向上: **{speedup:.1f}倍**
"""

report_path = f"{RESULTS_DIR}/baseline_report_{timestamp}.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)
print(f"✅ レポート保存: {report_path}")

---
## 🎉 完了

ベースラインRAGシステムの構築とテストが完了しました。

**次のステップ**:
1. Phase 5: 55件の高度なテストケース実装
2. Phase 6: 複数モデル比較
3. Phase 7: RAG改良
4. Phase 8: ファインチューニング